# Parameter Sensitivity Analysis (Continuous & Rule-Based)

Single notebook for both **continuous** and **rule-based** features. Set `feature_type` in
`feature_research/config.py` (or override below); the same workflow runs:

- **Continuous:** Sweeps feature params + `n_bins`, fits `ContinuousBinningModel`, computes metric on selected returns.
- **Rule-based:** Sweeps feature params only; feature is already a binary signal — no model fitting.

Both paths run neighbor smoothing and Marginal Peak Selection (MPS) → recommendations and plots.

**Workflow:**
1. Set `feature_type` in `feature_research/config.py` (CONTINUOUS or RULE_BASED); override inline if needed.
2. Run all cells (Kernel → Restart & Run All).
3. Interact with the Plotly figures in Section 5.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import replace

from feature_research.config import FeatureType
from feature_research.in_sample.config import load_config
from feature_research.in_sample.metric_helpers import compute_param_sensitivity_metric
from feature_research.in_sample.data_loader import (
    expand_bias_specs,
    load_features_for_combo,
    populate_cache_if_needed,
    param_combo_label,
)
from feature_selection.base_models.continuous_binning import ContinuousBinningModel
from eda.parameter_analysis import generate_parameter_sensitivity_report
from utils.core.enums import Ticker, TimeFrame

print("Imports OK")


Imports OK


## 1. Configuration

Edit `config.py` for persistent changes, or use the override cell below for one-off runs.
`dataclasses.replace()` is used because `ResearchConfig` is a frozen dataclass.

In [2]:
# --- Load defaults from config.py ---
config = load_config()

# ============================================================
# OVERRIDE CELL — edit here to customise without touching config.py
# ============================================================

# config = replace(config, tickers=[Ticker.ES])
# config = replace(config, start=datetime(2000, 1, 1), end=datetime(2015, 12, 31))
# Switch to rule-based without editing feature_research/config.py:
# config = replace(config, feature_type=FeatureType.RULE_BASED)
# config = replace(config,
#     bias_spec={
#         "module_name": "rsi",
#         "timeframes": [TimeFrame.D],
#         "params": {"lookback": [5, 10, 14, 20, 30]},
#     }
# )

# --- Param sensitivity: from config (edit feature_research/config.py to tune) ---
ps = config.param_sensitivity
STABILITY_THRESHOLD = ps.stability_threshold
# top_k is global (GlobalResearchDefaults); walkforward is built from it
TOP_K = config.walkforward.top_k
BINNING_SELECTION_METRIC = config.binning_params.selection_metric
METRIC_COL = BINNING_SELECTION_METRIC
PLOT_3D_MODE = ps.plot_3d_mode
# To change top_k, edit GlobalResearchDefaults in feature_research/config.py load_config().

# --- Dispatch on feature_type (from feature_research/config.py) ---
is_continuous = config.feature_type == FeatureType.CONTINUOUS
if is_continuous:
    N_BINS_GRID = list(config.binning_params.bin_counts)
    if not N_BINS_GRID:
        raise ValueError(
            "binning_params.bin_counts is empty. Provide at least one n_bins value "
            "for parameter sensitivity."
        )
else:
    N_BINS_GRID = None  # rule-based: fixed 3 levels, no n_bins sweep

# Derive varying param names (list-valued params only)
varying_params = [
    k for k, v in config.bias_spec["params"].items()
    if isinstance(v, list) and len(v) > 1
]
if not varying_params:
    print(
        "WARNING: No list-valued params found in bias_spec. "
        "Parameter sensitivity requires at least one param with multiple values. "
        "Results will be a single-point grid for feature params; only n_bins will vary."
    )
    varying_params = list(config.bias_spec["params"].keys())

bp = config.binning_params
single_bin = (
    is_continuous
    and bp.bin_index_max is not None
    and bp.bin_index_min == bp.bin_index_max
)
if single_bin:
    report_param_names = [*varying_params, "n_bins"]
elif is_continuous:
    report_param_names = [*varying_params, "n_bins", "selected_bin"]
else:
    report_param_names = varying_params

from feature_research.walkforward.marginal_peak_selection import MarginalPeakConfig
# marginal_dim=2: C(D,2) pairwise (interpretable); marginal_dim=3: C(D,3) 3D tables (better noise reduction for 5D+ grids)
MPS_CONFIG = MarginalPeakConfig(
    k_max=TOP_K,
    min_gap=ps.marginal_min_gap,
    min_cell_size=ps.marginal_min_cell_size,
    fallback_k=ps.marginal_fallback_k,
    marginal_dim=ps.marginal_dim,
)
print(f"Feature type : {config.feature_type.value}")
print(f"Module      : {config.bias_spec['module_name']}")
print(f"Tickers     : {[t.name for t in config.tickers]}")
print(f"Period      : {config.start.date()} -> {config.end.date()}")
print(f"Varying     : {varying_params}")
if is_continuous:
    print(f"n_bins grid : {N_BINS_GRID}")
print(f"METRIC      : {METRIC_COL}  |  THRESHOLD: {STABILITY_THRESHOLD}")
print(f"BIN METRIC  : {BINNING_SELECTION_METRIC}")
print(f"3D MODE     : {PLOT_3D_MODE}")
print(f"Selection   : method=marginal_peak  k_max={MPS_CONFIG.k_max}  min_gap={MPS_CONFIG.min_gap}  min_cell_size={MPS_CONFIG.min_cell_size}  marginal_dim={MPS_CONFIG.marginal_dim}")



Feature type : continuous
Module      : cyclical_rsi
Tickers     : ['ES', 'NQ']
Period      : 2000-01-01 -> 2016-12-31
Varying     : ['short_period', 'long_period', 'rsi_period']
n_bins grid : [12, 11, 10]
METRIC      : t_stat  |  THRESHOLD: 0.5
BIN METRIC  : t_stat
3D MODE     : surface_slices
Selection   : method=marginal_peak  k_max=1  min_gap=0.1  min_cell_size=2  marginal_dim=2


## 2. Feature Extraction

Loads feature + target data for every param combo. Uses the cache if `config.use_cache=True`.
Set `config.populate_cache=True` to auto-populate missing cache entries.

In [3]:
populate_cache_if_needed(config)

expanded_specs = expand_bias_specs(config.bias_spec)
print(f"Parameter grid: {len(expanded_specs)} combinations\n")

combo_data = {}
for spec in expanded_specs:
    result = load_features_for_combo(spec, config)
    if result is not None:
        feature, target, feature_col = result
        combo_data[param_combo_label(spec["params"])] = {
            "feature": feature,
            "target": target,
            "params": spec["params"],
        }
    else:
        print(f"  SKIP: {spec['params']}")

print(f"\nLoaded {len(combo_data)}/{len(expanded_specs)} combos successfully")

[data_loader] Populating cache with full OHLC range: 1997-09-09 -> 2025-09-19
  Adding required auxiliary: atr (for log_return_atr/log_return_ewsd)
  Adding required auxiliary: ewsd (for log_return_atr/log_return_ewsd)
Populating cache for 364 combinations...
  Bias node specs: 182 (including required atr/ewsd if needed)
  Tickers: 2
  Date range: 1997-09-09 00:00:00 to 2025-09-19 00:00:00
  Max workers: 4
  [1/364] cyclical_rsi / NQ / D
  [2/364] cyclical_rsi / ES / D
  [3/364] cyclical_rsi / ES / D
  [4/364] cyclical_rsi / NQ / D
  [5/364] cyclical_rsi / ES / D
  [6/364] cyclical_rsi / NQ / D
  [7/364] cyclical_rsi / ES / D
  [8/364] cyclical_rsi / NQ / D
  [9/364] cyclical_rsi / NQ / D
  [10/364] cyclical_rsi / ES / D
  [11/364] cyclical_rsi / ES / D
  [12/364] cyclical_rsi / NQ / D
  [13/364] cyclical_rsi / ES / D
  [14/364] cyclical_rsi / NQ / D
  [15/364] cyclical_rsi / ES / D
  [16/364] cyclical_rsi / NQ / D
  [17/364] cyclical_rsi / NQ / D
  [18/364] cyclical_rsi / ES / D
  [19

## 3. Metric Grid

- **Continuous:** Fits `ContinuousBinningModel` for each `(feature params, n_bins)` combo; then for each bin index in `0 .. n_bins-1` computes the config-defined metric on returns in that bin. Emits one row per `(feature params, n_bins, selected_bin)` so the grid is 3D — enabling stable-region selection to favor regions where bin 0 (tail/mean reversion) is strong.
- **Rule-based:** Feature is already a binary signal (1 = trade); computes the config-defined binning metric directly on returns where signal fires — no model fitting.

Builds `results_df` with `param{K}_value` columns (`param1`=varying params, `param2`=n_bins, `param3`=selected_bin for continuous) for `generate_parameter_sensitivity_report()`.


In [4]:
rows = []

for spec in expanded_specs:
    label = param_combo_label(spec["params"])
    if label not in combo_data:
        continue

    data = combo_data[label]
    feature = data["feature"]
    target = data["target"]

    if is_continuous:
        bp = config.binning_params
        bin_min = bp.bin_index_min
        bin_max = bp.bin_index_max
        single_bin = bp.bin_index_max is not None and bp.bin_index_min == bp.bin_index_max
        min_obs = 2 if single_bin else 5
        for n_bins in N_BINS_GRID:
            try:
                model = ContinuousBinningModel(n_bins=n_bins)
                model.fit(feature, target)
                bin_assignments = model._assign_bins(feature)
                bin_end = (bin_max + 1) if bin_max is not None else n_bins
                bin_range = range(bin_min, min(bin_end, n_bins))
                for bin_index in bin_range:
                    # ContinuousBinningModel uses base _assign_bins (np.digitize): 1-based;
                    # bin 0 = below first edge (empty), first data bin = 1.
                    model_bin = bin_index + 1
                    mask = (bin_assignments == model_bin)
                    selected_returns = target.loc[mask].values
                    if len(selected_returns) < min_obs:
                        continue
                    metric_value = compute_param_sensitivity_metric(
                        selected_returns, BINNING_SELECTION_METRIC
                    )
                    row = {}
                    for k, name in enumerate(report_param_names):
                        if name == "n_bins":
                            row[f"param{k+1}_value"] = n_bins
                        elif name == "selected_bin":
                            row[f"param{k+1}_value"] = bin_index
                        else:
                            row[f"param{k+1}_value"] = spec["params"][name]
                    row[METRIC_COL] = metric_value
                    rows.append(row)
            except Exception as e:
                print(f"  ERROR [{label}, n_bins={n_bins}]: {e}")
    else:
        try:
            signals = np.asarray(feature.values == 1).flatten()
            selected_returns = target.values[signals]

            if len(selected_returns) < 5:
                print(f"  SKIP [{label}]: too few signals ({len(selected_returns)}) to compute reliable metric")
                continue

            metric_value = compute_param_sensitivity_metric(selected_returns, BINNING_SELECTION_METRIC)
            row = {f"param{k+1}_value": spec["params"][key] for k, key in enumerate(varying_params)}
            row[METRIC_COL] = metric_value
            rows.append(row)

        except Exception as e:
            print(f"  ERROR [{label}]: {e}")

results_df = pd.DataFrame(rows)
print(f"Grid results: {len(results_df)} rows")
results_df


Grid results: 540 rows


,param1_value,param2_value,param3_value,param4_value,t_stat
0,3,40,2,12,3.201643
1,3,40,2,11,2.676588
2,3,40,2,10,2.925112
3,3,40,4,12,2.236308
4,3,40,4,11,2.693235
...,...,...,...,...,...
535,8,140,7,11,0.829829
536,8,140,7,10,0.943025
537,8,140,10,12,0.958646
538,8,140,10,11,1.325900


## 4. Parameter Sensitivity Report

Runs: neighbor smoothing → stable region detection → recommendations.

In [5]:
if results_df.empty:
    msg = (
        f"No results collected. Combos loaded: {len(combo_data)}/{len(expanded_specs)}. "
        f"Ensure cache is available or set config.populate_cache=True."
    )
    if is_continuous and hasattr(config, "binning_params"):
        bp = config.binning_params
        msg += (
            f" With bin_index_min={bp.bin_index_min}, bin_index_max={bp.bin_index_max}, "
            "each selected bin must have at least 2 (single-bin) or 5 (multi-bin) observations. "
            "Try increasing bin_index_max or using a larger dataset."
        )
    raise RuntimeError(msg)

report = generate_parameter_sensitivity_report(
    results_df=results_df,
    param_names=report_param_names,
    metric_col=METRIC_COL,
    stability_threshold=STABILITY_THRESHOLD,
    top_k=TOP_K,
    plot_3d_mode=PLOT_3D_MODE,
    marginal_peak_config=MPS_CONFIG,
)

print(f"Module              : {config.bias_spec['module_name']}")
print(f"Metric              : {report.metric_name}")
print(f"Grid size           : {report.n_parameter_combinations}")
print(f"Stable regions      : {report.n_stable_regions}")
print(f"Mean stability ratio: {report.mean_stability_ratio:.3f}")
print(f"% stable combos     : {report.pct_stable_combinations:.1%}")
if report.mps_result is not None:
    r = report.mps_result
    print(f"MPS selected pair : {r.selected_pair}")
    print(f"MPS peak cell     : {r.peak_cell}")
    print(f"MPS gap           : {r.gap:.4f}  fallback={r.fallback_used}")
print(f"\nTop-{TOP_K} recommendations:")
for i, combo in enumerate(report.top_k_combinations, 1):
    params_str = ", ".join(
        f"{name}={val}" for name, val in zip(report_param_names, combo)
    )
    print(f"  {i}. {params_str}")




Module              : cyclical_rsi
Metric              : t_stat
Grid size           : 540
Stable regions      : 0
Mean stability ratio: 1.018
% stable combos     : 0.2%
MPS selected pair : ('rsi_period', 'short_period')
MPS peak cell     : (np.float64(2.0), np.float64(4.0))
MPS gap           : 0.4458  fallback=False

Top-1 recommendations:
  1. short_period=4.0, long_period=120.0, rsi_period=2.0, n_bins=10.0


In [6]:
if report.marginal_tables:
    print(f"Marginal Peak Selection — {len(report.marginal_tables)} pairwise tables")
    if report.mps_result is not None:
        r = report.mps_result
        print(f"  selected_pair={r.selected_pair}  peak_cell={r.peak_cell}  gap={r.gap:.4f}  fallback_used={r.fallback_used}")
    for (dim_a, dim_b), tbl in report.marginal_tables.items():
        print(f"\n--- ({dim_a}, {dim_b}) ---")
        display(tbl)

Marginal Peak Selection — 6 pairwise tables
  selected_pair=('rsi_period', 'short_period')  peak_cell=(np.float64(2.0), np.float64(4.0))  gap=0.4458  fallback_used=False

--- (long_period, n_bins) ---


,long_period,n_bins,cell_mean,cell_size
0,40.0,12.0,2.812209,30
1,40.0,11.0,2.806970,30
2,40.0,10.0,2.828014,30
3,60.0,12.0,3.045230,30
4,60.0,11.0,3.003792,30
5,60.0,10.0,2.870342,30
6,80.0,12.0,2.992887,30
7,80.0,11.0,2.993569,30
8,80.0,10.0,2.945751,30
9,100.0,12.0,2.943030,30



--- (long_period, rsi_period) ---


,long_period,rsi_period,cell_mean,cell_size
0,40.0,2.0,3.073091,15
1,40.0,4.0,3.118749,15
2,40.0,5.0,3.038883,15
3,40.0,6.0,2.822396,15
4,40.0,7.0,2.682463,15
5,40.0,10.0,2.158804,15
6,60.0,2.0,3.484667,15
7,60.0,4.0,3.293099,15
8,60.0,5.0,3.157394,15
9,60.0,6.0,2.987638,15



--- (long_period, short_period) ---


...

,n_bins,rsi_period,cell_mean,cell_size
0,12.0,2.0,3.540462,30
1,11.0,2.0,3.661817,30
2,10.0,2.0,3.678601,30
3,12.0,4.0,3.281415,30
4,11.0,4.0,3.268858,30
5,10.0,4.0,3.221580,30
6,12.0,5.0,3.065149,30
7,11.0,5.0,2.976569,30
8,10.0,5.0,2.949760,30
9,12.0,6.0,2.734983,30



--- (n_bins, short_period) ---


,n_bins,short_period,cell_mean,cell_size
0,12.0,3.0,2.933660,36
1,11.0,3.0,2.993182,36
2,10.0,3.0,2.978992,36
3,12.0,4.0,3.767532,36
4,11.0,4.0,3.688321,36
5,10.0,4.0,3.635890,36
6,12.0,5.0,3.113962,36
7,11.0,5.0,3.055206,36
8,10.0,5.0,2.948688,36
9,12.0,6.0,2.684626,36



--- (rsi_period, short_period) ---


,rsi_period,short_period,cell_mean,cell_size
0,2.0,3.0,3.487293,18
1,4.0,3.0,2.943229,18
2,5.0,3.0,2.839684,18
3,6.0,3.0,2.845967,18
4,7.0,3.0,2.870012,18
5,10.0,3.0,2.825481,18
6,2.0,4.0,4.548634,18
7,4.0,4.0,4.102877,18
8,5.0,4.0,3.941712,18
9,6.0,4.0,3.694839,18


### Stable regions and summary statistics

Summary stats from the selection algorithm and a table of all identified stable regions (contiguous grid regions above the stability threshold). With MPS selection, stable regions are not computed; the table below is empty.

In [7]:
# --- Summary statistics ---
grid = report.grid_results
ratios = grid["stability_ratio"].dropna()
raw_metric = report.metric_name
smoothed_metric = f"smoothed_{raw_metric}"
obj_vals = grid[raw_metric].dropna() if raw_metric in grid.columns else pd.Series(dtype=float)
smoothed_vals = grid[smoothed_metric].dropna() if smoothed_metric in grid.columns else pd.Series(dtype=float)

summary = pd.DataFrame([
    {"Metric": "Objective metric", "Value": report.metric_name},
    {"Metric": "Stability threshold", "Value": report.stability_threshold},
    {"Metric": "Grid size (param combos)", "Value": report.n_parameter_combinations},
    {"Metric": "Stable regions", "Value": report.n_stable_regions},
    {"Metric": "Total combos in stable regions", "Value": sum(r.n_combinations for r in report.stable_regions)},
    {"Metric": "Mean stability ratio", "Value": f"{report.mean_stability_ratio:.3f}"},
    {"Metric": "Median stability ratio", "Value": f"{report.median_stability_ratio:.3f}"},
    {"Metric": "Std stability ratio", "Value": f"{ratios.std():.3f}" if len(ratios) > 1 else "—"},
    {"Metric": "Min / Max stability ratio", "Value": f"{ratios.min():.3f} / {ratios.max():.3f}" if len(ratios) > 0 else "—"},
    {"Metric": "% combos in stable regions", "Value": f"{report.pct_stable_combinations:.1%}"},
    {"Metric": f"Best raw {raw_metric}", "Value": f"{obj_vals.max():.4f}" if len(obj_vals) > 0 else "—"},
    {"Metric": f"Best smoothed {raw_metric}", "Value": f"{smoothed_vals.max():.4f}" if len(smoothed_vals) > 0 else "—"},
    {"Metric": f"25th / 75th pct raw {raw_metric}", "Value": f"{obj_vals.quantile(0.25):.4f} / {obj_vals.quantile(0.75):.4f}" if len(obj_vals) > 0 else "—"},
    {"Metric": f"25th / 75th pct smoothed {raw_metric}", "Value": f"{smoothed_vals.quantile(0.25):.4f} / {smoothed_vals.quantile(0.75):.4f}" if len(smoothed_vals) > 0 else "—"},
]).set_index("Metric")
display(summary)

# --- All stable regions (param ranges, size, objective stats) ---
if not report.stable_regions:
    print("No stable regions with ≥2 combinations identified.")
else:
    region_rows = [
        {
            "region": i + 1,
            "param_ranges": ", ".join(
                f"{report_param_names[k]}={r.param_ranges[f'param{k+1}'][0]}..{r.param_ranges[f'param{k+1}'][1]}"
                for k in range(len(report_param_names))
            ),
            "n_combinations": r.n_combinations,
            "mean_objective": round(r.mean_objective, 4),
            "min_objective": round(r.min_objective, 4),
            "max_objective": round(r.max_objective, 4),
            "mean_stability_ratio": round(r.mean_stability_ratio, 3),
            "is_boundary": r.is_boundary_region,
        }
        for i, r in enumerate(report.stable_regions)
    ]
    regions_df = pd.DataFrame(region_rows)
    display(regions_df)

    # --- Per-region param combos with per-combo stats (raw, smoothed, stability_ratio, n_neighbors) ---
    param_cols = [f"param{k+1}_value" for k in range(len(report_param_names))]
    smoothed_col = f"smoothed_{report.metric_name}"
    print("\nParam combos per region (with stats from grid):")
    for i, r in enumerate(report.stable_regions):
        combos_df = pd.DataFrame(
            [list(t) for t in r.param_combinations],
            columns=report_param_names,
        )
        merged = combos_df.merge(
            report.grid_results,
            left_on=report_param_names,
            right_on=param_cols,
            how="left",
        )
        stats_cols = [report.metric_name, smoothed_col, "stability_ratio", "n_neighbors"]
        keep = report_param_names + [c for c in stats_cols if c in merged.columns]
        out = merged[keep].copy()
        round_map = {c: 4 for c in [report.metric_name, smoothed_col] if c in out.columns}
        if "stability_ratio" in out.columns:
            round_map["stability_ratio"] = 3
        if round_map:
            out = out.round(round_map)
        renames = {}
        if report.metric_name in out.columns:
            renames[report.metric_name] = f"{report.metric_name}_raw"
        if smoothed_col in out.columns:
            renames[smoothed_col] = f"{report.metric_name}_smoothed"
        if renames:
            out = out.rename(columns=renames)
        print(f"\nRegion {i + 1} ({r.n_combinations} combos):")
        display(out)

,Value
Metric,
Objective metric,t_stat
Stability threshold,0.5
Grid size (param combos),540
Stable regions,0
Total combos in stable regions,0
Mean stability ratio,1.018
Median stability ratio,1.001
Std stability ratio,0.094
Min / Max stability ratio,0.806 / 1.613


No stable regions with ≥2 combinations identified.


## 5. Interactive Visualizations

The appropriate plot is selected automatically based on grid dimensionality:
- 1D grid → raw + smoothed line with shaded stable regions
- 2D grid → heatmap with stability contour overlay
- 3D+ grid → heatmap slices with dropdown + slider controls

In [8]:
if report.plot_1d is not None:
    report.plot_1d.show()

if report.plot_2d is not None:
    report.plot_2d.show()

if report.plot_3d is not None:
    report.plot_3d.show()

## 6. Optional Save

Uncomment cells below to save plots as interactive HTML or grid results as parquet.

In [9]:
# --- OPTIONAL: Save outputs (uncomment to enable) ---

# from pathlib import Path
# save_dir = config.reports_dir / "param_sensitivity"
# save_dir.mkdir(parents=True, exist_ok=True)

# if report.plot_1d is not None:
#     report.plot_1d.write_html(save_dir / f"param_sensitivity_1d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 1D plot -> {save_dir}")

# if report.plot_2d is not None:
#     report.plot_2d.write_html(save_dir / f"param_sensitivity_2d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 2D plot -> {save_dir}")

# if report.plot_3d is not None:
#     report.plot_3d.write_html(save_dir / f"param_sensitivity_3d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 3D plot -> {save_dir}")

# report.grid_results.to_parquet(save_dir / "param_sensitivity_grid.parquet")
# print(f"Saved grid results -> {save_dir / 'param_sensitivity_grid.parquet'}")